# Import packages

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import warnings

warnings.filterwarnings('ignore')


## 📌 2. Load Raw Dataset

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kanchana1990/algorithmic-trading-macro-stress-and-asset-regimes")

print("Path to dataset files:", path)

100%|██████████| 374k/374k [00:00<00:00, 67.1MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/kanchana1990/algorithmic-trading-macro-stress-and-asset-regimes/versions/1


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Global_Market_Stress_and_Liquidity_Regimes.csv to Global_Market_Stress_and_Liquidity_Regimes.csv


In [7]:
df = pd.read_csv('Global_Market_Stress_and_Liquidity_Regimes.csv')

df.tail()

,Date,Equities_US,Equities_Tech,Equities_Emerging,Bonds_LongTerm,Gold,Oil,Volatility_Index,Crypto_Bitcoin,Yield_Curve_Spread,High_Yield_Spread,Financial_Stress_Index,SPY_Drawdown,SPY_Rolling_Vol_30d,BTC_Rolling_Vol_30d,Stock_Bond_Corr_90d,SPY_RSI_14,GLD_RSI_14
4145,2026-02-21,689.429993,608.809998,62.340000,89.410004,468.619995,80.849998,19.090000,68003.765625,0.60,2.86,-0.6208,-0.008713,0.095746,0.794547,0.159538,53.364001,57.476338
4146,2026-02-22,689.429993,608.809998,62.340000,89.410004,468.619995,80.849998,19.090000,67659.390625,0.60,2.86,-0.6208,-0.008713,0.095741,0.794018,0.128982,53.364001,57.476338
4147,2026-02-23,682.390015,601.409973,61.650002,89.739998,481.279999,80.900002,21.010000,64616.738281,0.60,2.95,-0.6208,-0.018836,0.100235,0.803970,0.099189,42.986401,63.198055
4148,2026-02-24,687.349976,607.869995,62.619999,89.900002,474.609985,80.760002,19.549999,64616.738281,0.61,2.95,-0.6208,-0.011704,0.102617,0.801768,0.091355,50.317126,58.715496
4149,2026-02-25,687.349976,607.869995,62.619999,89.900002,474.609985,80.760002,19.549999,65568.492188,0.61,2.95,-0.6208,-0.011704,0.101452,0.799781,0.091355,50.317126,58.715496


# Create acategorical 'Market Regime' bases on macro indicators
### This is a classic quant approach to labeling time.series data

In [8]:
conditions = [
    # Risk Off / Crisis: High Financial Stress and plunging equities
    (df['Financial_Stress_Index'] > 1.0 ) & (df['SPY_Drawdown'] < -0.10),

    #Inflationary / Rate Shock: Rising Yields, negative Stock/Bond correlation breaking down
    (df['Yield_Curve_Spread'] < 0) & (df['Stock_Bond_Corr_90d'] > 0),

    # Risk on / Expansion: Low stress, normal yield curve
    (df['Financial_Stress_Index'] < 0) & (df['SPY_Drawdown'] > -0.05)
]

choices = ['Crisis / Risk-Off', 'Rate Shock / Inversion', 'Expansion / Risk-On']

df['Macro_Regime'] = np.select(conditions, choices, default='Transition / Mixed')

print("Regimen Distribution (Trading Days):")
print(df['Macro_Regime'].value_counts())

Regimen Distribution (Trading Days):
Macro_Regime
Expansion / Risk-On       2275
Transition / Mixed        1205
Rate Shock / Inversion     596
Crisis / Risk-Off           74
Name: count, dtype: int64


In [9]:
# Build the Master Institutional Dashboard
fig = make_subplots (
    rows=4, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    subplot_titles=("S&P 500 Price & Drawdown (SPY)",
                    "Systemic Risk: Financial Stress & High Yirld Spreads",
                    "Liquididy: 10Y-2Y Yield Curve Spread",
                    "Regime Indicator: 90-Day Stock/Bond Correlation")
)

fig.add_trace(go.Scatter(x=df.index, y=df['Equities_US'], name="SPY Price", line=dict(color='#00ffcc', width=2)), row=1, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['Financial_Stress_Index'], name="Fed Stress Index", fill='tozeroy', line=dict(color='#ff3366', width=1)), row=2, col=1)
fig.add_trace(go.Scatter(x=df.index, y=df['High_Yield_Spread'], name="High Yield Spread", line=dict(color='#ff9900', width=2)), row=2, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['Yield_Curve_Spread'], name="Yield Curve Spread", fill='tozeroy', line=dict(color='#33ccff', width=1.5)), row=3, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=3, col=1)

fig.add_trace(go.Scatter(x=df.index, y=df['Stock_Bond_Corr_90d'], name="Stock/Bond Corr", line=dict(color='#cc33ff', width=2)), row=4, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=4, col=1)

fig.update_layout(
    title="<b>Global Market Liquidity & Regime Dashboard</b>",
    height=1000,
    template="plotly_dark",
    hovermode="x unified",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

## Cross-Asset Capital Flows

In the final section, we look ar how capital rotates between Risk Assets(Tech, Bitcoin) and Safe Havens(Bonds, Gold, Dollar). By generating an interactive correlation matrix, you can quickly identify the diversification benefits of different asset classes during the timeframe provided.

In [12]:
# Select only the core asset price columns for the correlation matrix
assets_only = df[['Equities_US', 'Equities_Tech', 'Equities_Emerging',
                  'Bonds_LongTerm', 'Gold', 'Oil', 'Crypto_Bitcoin']]

# Calculate the daily returns, then find the correlation
asset_returns = assets_only.pct_change().dropna()
corr_matrix = asset_returns.corr().round(3)

# Build interactive heatmap
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect="auto",
    color_continuous_scale='RdBu_r',
    title="<b>Interactive Cross-Asset Correlation Matrix (Daily Returns)</b>"
)

fig_corr.update_layout(
    template="plotly_dark",
    height=600,
    xaxis_title="Asset Class",
    yaxis_title="Asset Class"
)

fig_corr.show()

In [14]:
#Quant Summmary Dashboard
# Create a comprehensive, asymmetric dashboard layout
fig_summary = make_subplots(
    rows=3, cols=2,
    specs=[[{"colspan": 2}, None],  # Top row spans both columns
           [{"type": "xy"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "box"}]], # Bottom right is a box/violin plot
    subplot_titles=(
        "1. Cumulative Wealth Growth (Base 100, Log Scale)",
        "2. Volatility Regimes: S&P 500 vs Bitcoin",
        "3. Momentum Tracking (14-Day RSI)",
        "4. Systemic Risk: Stress vs. S&P 500 Drawdown",
        "5. Drawdown Distributions"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# --- PANEL 1: Cumulative Returns (Top Row, Spans 2 Columns) ---
# Calculate cumulative returns from normalized daily percentage changes
assets = ['Equities_US', 'Equities_Tech', 'Bonds_LongTerm', 'Gold', 'Crypto_Bitcoin']
df_returns = df[assets].pct_change().dropna()
df_cum = (1 + df_returns).cumprod() * 100

colors = {'Equities_US': '#00ffcc', 'Equities_Tech': '#3399ff',
          'Bonds_LongTerm': '#ff99cc', 'Gold': '#ffcc00', 'Crypto_Bitcoin': '#ff3366'}

for col in assets:
    fig_summary.add_trace(
        go.Scatter(x=df_cum.index, y=df_cum[col], name=col,
                   line=dict(color=colors[col], width=1.5)),
        row=1, col=1
    )
# Set log scale for crypto-distorted returns
fig_summary.update_yaxes(type="log", title_text="Growth (Log Scale)", row=1, col=1)

# --- PANEL 2: Volatility Comparison (Middle Row, Left) ---
fig_summary.add_trace(
    go.Scatter(x=df.index, y=df['SPY_Rolling_Vol_30d'], name="SPY 30d Vol",
               line=dict(color='#00ffcc', width=1)), row=2, col=1)
fig_summary.add_trace(
    go.Scatter(x=df.index, y=df['BTC_Rolling_Vol_30d'], name="BTC 30d Vol",
               line=dict(color='#ff3366', width=1, dash='dot')), row=2, col=1)
fig_summary.update_yaxes(title_text="Annualized Volatility", row=2, col=1)

# --- PANEL 3: Momentum / RSI (Middle Row, Right) ---
fig_summary.add_trace(
    go.Scatter(x=df.index, y=df['SPY_RSI_14'], name="SPY RSI",
               line=dict(color='#3399ff', width=1)), row=2, col=2)
fig_summary.add_trace(
    go.Scatter(x=df.index, y=df['GLD_RSI_14'], name="Gold RSI",
               line=dict(color='#ffcc00', width=1)), row=2, col=2)
# Add Overbought/Oversold lines
fig_summary.add_hline(y=70, line_dash="dash", line_color="red", row=2, col=2)
fig_summary.add_hline(y=30, line_dash="dash", line_color="green", row=2, col=2)
fig_summary.update_yaxes(title_text="RSI Value", row=2, col=2)

# --- PANEL 4: Macro Stress Scatter (Bottom Row, Left) ---
# Filter out NaNs for clean scatter
df_valid = df[['Financial_Stress_Index', 'SPY_Drawdown']].dropna()
fig_summary.add_trace(
    go.Scatter(x=df_valid['Financial_Stress_Index'], y=df_valid['SPY_Drawdown'],
               mode='markers', marker=dict(color=df_valid['SPY_Drawdown'],
               colorscale='RdYlGn', showscale=False, size=4, opacity=0.6),
               name="Stress vs Drawdown"), row=3, col=1)
fig_summary.update_xaxes(title_text="Fed Financial Stress Index", row=3, col=1)
fig_summary.update_yaxes(title_text="SPY Drawdown %", row=3, col=1)

# --- PANEL 5: Drawdown Distributions (Bottom Row, Right) ---
# Create box plots to show the fat-tail risk of different regimes
fig_summary.add_trace(
    go.Box(y=df['SPY_Drawdown'].dropna(), name="SPY Drawdowns",
           marker_color='#00ffcc'), row=3, col=2)
fig_summary.add_trace(
    go.Box(y=df['Stock_Bond_Corr_90d'].dropna(), name="Stock/Bond Corr",
           marker_color='#cc33ff'), row=3, col=2)

# --- FINAL LAYOUT ADJUSTMENTS ---
fig_summary.update_layout(
    title="<b>The Quant Tear Sheet: Macro, Momentum, and Risk</b>",
    height=1400, # Massive height to fit everything cleanly
    template="plotly_dark",
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    hovermode="closest"
)

# [Image of a dark mode quantitative finance dashboard showing cumulative returns, volatility metrics, and scatter plots for asset regimes]

fig_summary.show()